# Homework 1 — Recurrent Networks (RNN / LSTM / GRU)

| | |
|:---|:---|
| **Course** | Large Language Models |
| **Submission** | This file `hw1.ipynb` only, with **Run All** outputs saved |
| **Full assignment text** | `40404084_HW1.pdf` |

**Data files** (put next to the notebook, or edit paths in the imports cell):

- `Book.txt` — Question 2  
- `dictionary.txt`, `sentiment_labels.txt` — Question 3

<div style="direction:rtl">
آرمیتا اصحاب یمین

40404084


In [1]:
from __future__ import annotations

import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from gensim.models import Word2Vec
from tensorflow.keras import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Masking, Activation
from tensorflow.keras.optimizers import RMSprop, Adam

# --- reproducibility ---
tf.keras.utils.set_random_seed(42)
np.random.seed(42)
random.seed(42)

# --- data paths (edit if your files live elsewhere) ---
BASE = Path.cwd()
BOOK_PATH = BASE / "Book.txt"
DICT_PATH = BASE / "dictionary.txt"
LABELS_PATH = BASE / "sentiment_labels.txt"
print("Working directory:", BASE)

c:\Users\SamRayaneh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Working directory: f:\esfahan\LLM\HW1


---
## Question 1 — Patterns (`a^k N b^k` and `(ab)^k N (ab)^k`)

**Vocabulary:** symbols `a`, `b`, `N`, and an **end** marker. The PDF calls it `"end"`; in code you must use **one character** (e.g. `E`) so the one-hot size is 4.

### Deliverables (parts a & b)
- Build the training set for **all k &lt; 11** (k = 1 … 10) for the first pattern.
- Train an **LSTM with 10 hidden units** on (prefix, next character) pairs with one-hot inputs, **fixed length 40**, **left** padding.
- Print `model.summary()` and training logs (loss / accuracy).
- **(b)** For several values of **k** (including at least one **k &gt; 10**), run autoregressive generation and **report** how well the model matches the pattern.

### Deliverables (part c)
- Same setup for `(ab)^k N (ab)^k` with **LSTM, 20 hidden units**, input length **80**, **k = 1 … 15**.

**Complete the code cells below.**

In [12]:
# -----------------------------------------------------------------------------
#  Question 1 — vocabulary & helpers (modified internals only)
# -----------------------------------------------------------------------------

CHARS = ["a", "b", "N", "E"]
char_to_idx = {c: i for i, c in enumerate(CHARS)}
VOCAB = len(CHARS)


def anb_full_string(k: int) -> str:
    """Full string for a^k N b^k plus the end token (one char)."""
    left = "".join(["a" for _ in range(k)])
    right = "".join(["b" for _ in range(k)])
    return f"{left}N{right}E"


def abab_full_string(k: int) -> str:
    """Full string for (ab)^k N (ab)^k plus the end token."""
    seg = "".join("ab" for _ in range(k))
    return seg + "N" + seg + "E"


def prefix_xy_pairs(full: str):
    """Return (prefixes, next_chars) for next-character prediction."""
    prefixes = []
    next_chars = []
    for idx in range(1, len(full)):
        prefixes.append(full[:idx])
        next_chars.append(full[idx])
    return prefixes, next_chars


def strings_to_xy_onehot(prefixes, next_chars, max_len: int):
    """
    X: (batch, max_len, VOCAB), left-padded one-hot prefixes.
    y: (batch, VOCAB) one-hot next character.
    """
    batch = len(prefixes)
    X = np.zeros((batch, max_len, VOCAB), dtype=np.float32)
    y = np.zeros((batch, VOCAB), dtype=np.float32)

    for row, (pref, nxt) in enumerate(zip(prefixes, next_chars)):
        trimmed = pref[-max_len:]
        pad = max_len - len(trimmed)

        for col, ch in enumerate(trimmed):
            idx = char_to_idx[ch]
            X[row, pad + col, idx] = 1.0

        y[row, char_to_idx[nxt]] = 1.0

    return X, y



def build_anb_dataset(k_max: int, max_len: int):
    """Build (X, y) for the a^k N b^k family, k = 1 .. k_max."""
    prefixes = []
    next_chars = []

    for k in range(1, k_max + 1):
        full = anb_full_string(k)
        p_list, n_list = prefix_xy_pairs(full)
        prefixes += p_list
        next_chars += n_list

    return strings_to_xy_onehot(prefixes, next_chars, max_len)



def build_abab_dataset(k_max: int, max_len: int):
    """Build (X, y) for the (ab)^k N (ab)^k family, k = 1 .. k_max."""
    prefixes = []
    next_chars = []

    for k in range(1, k_max + 1):
        full = abab_full_string(k)
        p_list, n_list = prefix_xy_pairs(full)
        prefixes.extend(p_list)
        next_chars.extend(n_list)

    return strings_to_xy_onehot(prefixes, next_chars, max_len)



def make_lstm_classifier(hidden: int, max_len: int) -> tf.keras.Model:
    """LSTM -> Dense(VOCAB) -> softmax; categorical_crossentropy."""
    layers = [
        LSTM(hidden, input_shape=(max_len, VOCAB)),
        Dense(VOCAB, activation="softmax")
    ]
    model = Sequential(layers)

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model



def greedy_generate(model, seed: str, max_len: int, max_steps: int = 200) -> str:
    """Autoregressive generation until end token E or max_steps."""
    result = seed

    for _ in range(max_steps):
        X = np.zeros((1, max_len, VOCAB), dtype=np.float32)

        cur = result[-max_len:]
        pad = max_len - len(cur)

        for j, ch in enumerate(cur):
            X[0, pad + j, char_to_idx[ch]] = 1.0

        probs = model.predict(X, verbose=0)[0]
        idx = int(np.argmax(probs))
        ch = CHARS[idx]

        result += ch
        if ch == "E":
            break

    return result

In [13]:
# -----------------------------------------------------------------------------
#  Question 1 (a,b): k_max = 10, max_len = 40
# -----------------------------------------------------------------------------
MAX_LEN_A = 40
# X_train, y_train = build_anb_dataset(...)
X_train, y_train = build_anb_dataset(k_max=10, max_len=MAX_LEN_A)

# model1 = make_lstm_classifier(...)
model1 = make_lstm_classifier(hidden=10, max_len=MAX_LEN_A)
print(model1.summary())

# model1.fit(...)
history1 = model1.fit(
    X_train, 
    y_train,
    epochs=20,
    batch_size=32,
)

c:\Users\SamRayaneh\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 10)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │            44 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 644 (2.52 KB)

 Trainable params: 644 (2.52 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0167 - loss: 1.4742  
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0083 - loss: 1.4539    
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0000e+00 - loss: 1.4344
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0000e+00 - loss: 1.4155
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0333 - loss: 1.3971
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1000 - loss: 1.3791
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2417 - loss: 1.3613
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4000 - loss: 1.3436
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6833 - loss: 1.3260
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7250 - loss: 1.3084
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7750 - loss: 1.2905
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.80


<div style="direction:rtl">
به دلیل این که داریم از الگوریتم حریصانه استفاده میکنیم و شبکه بسیار کوچک است و به اندازه ی کافی داده دریافت نکرده، نمیتوانیم انتظار داشته باشیم که به درستی پیشبینی کند. همچنین به دلیل انتخاب بیشترین احتمال شبکه در یک حلقه ی بی نهایت در پرینت b گیر می کند تا مرحله ای که به حد اندازه ی جمله برسد.
می توانیم با گذاشتن یک شمارنده این مسِئله را حل کنیم و خروجی را بعد از مرحله ای که تعداد bها از aها بیشتر شد متوقف کنیم.
شبه کد این قسمت را اینجا گذاشته ام و قبلا تست کرده ام و به خوبی کار می کند اما به دلیل این که حس کردم ممکن است تغییر اجباری الگوریتم حریصانه محسوب شود، در کد اصلی بارگذاری نکرده ام.

<div style="direction:ltr">

def greedy_generate_fixed(model, seed: str, max_len: int, max_steps: int = 200) -> str:

    """greedy decoding with a  b-limit to avoid infinite b loops."""
    result = seed

    for _ in range(max_steps):
        X = np.zeros((1, max_len, VOCAB), dtype=np.float32)

        cur = result[-max_len:]
        pad = max_len - len(cur)

        for j, ch in enumerate(cur):
            X[0, pad + j, char_to_idx[ch]] = 1.0

        probs = model.predict(X, verbose=0)[0]
        idx = int(np.argmax(probs))
        ch = CHARS[idx]

        # the change:
        if "N" in result:
            before_N, after_N = result.split("N", 1)
            a_count = before_N.count("a")
            b_count = after_N.count("b")

            # If model begins generating too many b's → terminate with E
            if b_count >= a_count:
                ch = "E"
        # -----------------------------------------

        result += ch
        if ch == "E":
            break

    return result

In [14]:
# -----------------------------------------------------------------------------
#  Question 1 (b): evaluate on several k (include at least one k > 10)
# -----------------------------------------------------------------------------
for k in [2,5,10,15]:
    seed = "a"*k + "N"
    print("k =", k)
    out = greedy_generate(model1, seed, MAX_LEN_A)
    print("Actual_Output:", out)
    print("Expected_Output:", "a"*k + "N" + "b"*k + "E")
    print("-----")
    


k = 2
Actual_Output: aaNbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb
Expected_Output: aaNbbE
-----
k = 5
Actual_Output: aaaaaNbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb
Expected_Output: aaaaaNbbbbbE
-----
k = 10
Actual_Output: aaaaaaaaaaNbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb
Expected_Output: aaaaaaaaaaNbbbbbbbbbbE
-----
k = 15
Actual_Output: aaaaaaaaaaaaaaaNbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb

In [15]:
# -----------------------------------------------------------------------------
#  Question 1 (c): k_max = 15, max_len = 80, LSTM hidden = 20
# -----------------------------------------------------------------------------
MAX_LEN_C = 80

Xc, yc = build_abab_dataset(k_max=15, max_len=MAX_LEN_C)

model2 = make_lstm_classifier(hidden=20, max_len=MAX_LEN_C)

print(model2.summary())

history2 = model2.fit(
    Xc,
    yc,
    epochs=25,
    batch_size=32,
)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 20)             │         2,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │            84 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,084 (8.14 KB)

 Trainable params: 2,084 (8.14 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4162 - loss: 1.3553
Epoch 2/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4848 - loss: 1.1871
Epoch 3/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4848 - loss: 1.0209
Epoch 4/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5515 - loss: 0.9149
Epoch 5/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8909 - loss: 0.8705
Epoch 6/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9152 - loss: 0.8397
Epoch 7/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9394 - loss: 0.8132
Epoch 8/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9394 - loss: 0.7871
Epoch 9/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9394 - loss: 0.7589
Epoch 10/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9394 - loss: 0.7272
Epoch 11/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9394 - loss: 0.6907
Epoch 12/25
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accu

In [16]:
# -----------------------------------------------------------------------------
#  Question 1 (b): evaluate on several k (include at least one k > 10)
# -----------------------------------------------------------------------------
for k in [2,5,10,15]:
    seed = "a"*k + "N"
    print("k =", k)
    out = greedy_generate(model2, seed, MAX_LEN_A)
    print("Actual_Output:", out)
    print("Expected_Output:", "a"*k + "N" + "b"*k + "E")
    print("-----")

k = 2
Actual_Output: aaNabababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababab
Expected_Output: aaNbbE
-----
k = 5
Actual_Output: aaaaaNabababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababab
Expected_Output: aaaaaNbbbbbE
-----
k = 10
Actual_Output: aaaaaaaaaaNabababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababab
Expected_Output: aaaaaaaaaaNbbbbbbbbbbE
-----
k = 15
Actual_Output: aaaaaaaaaaaaaaaNabababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababababa

---
## Question 2 — Text Generation

### GRU (short intro)

**GRU (Gated Recurrent Unit)** is a recurrent architecture with **update** and **reset** gates; it often uses **fewer parameters** than a classic LSTM (two gates vs three) and trains faster while staying competitive on many sequence tasks. At each step it blends the previous state with new input so longer dependencies can be captured more stably.

**Reference:** Junyoung Chung, Caglar Gulcehre, Kyunghyun Cho, Yoshua Bengio — *Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling* — [arXiv:1412.3555](https://arxiv.org/abs/1412.3555)

---

### Deliverables
- **(a)** From `Book.txt`: each input is **40** characters, target is the **next** character; consecutive windows overlap by **35** → stride **5**.
- **(b)** **GRU(128)**, **RMSProp(lr=0.01)**, **softmax** over the character vocabulary, train for at least **20 epochs**.
- Pick a **40-character** seed (e.g. **20** chars from a coherent span + **20** following chars), then continue sampling until the output has **400** characters; report **word count** and how many generated words appear in the book’s vocabulary (define “word”, e.g. with a letter regex).

**Implement in the cells below.**

In [17]:
def load_book(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    return text


def book_to_windows(text: str, seq_len: int = 40, overlap: int = 35):
    """
    Build sliding windows: step = seq_len - overlap (here: 5).
    Returns:
        X: (N, seq_len, vocab_size) one-hot
        y: (N, vocab_size) one-hot next character
        char_indices, indices_char, vocab_size
    """

    step = seq_len - overlap

    unique_chars = sorted(set(text))
    vocab_size = len(unique_chars)

    char_indices = dict((c, i) for i, c in enumerate(unique_chars))
    indices_char = dict((i, c) for i, c in enumerate(unique_chars))

    sentences = []
    next_chars = []

    limit = len(text) - seq_len
    for start in range(0, limit, step):
        end = start + seq_len
        sentences.append(text[start:end])
        next_chars.append(text[end])

    n_samples = len(sentences)

    X = np.zeros((n_samples, seq_len, vocab_size), dtype=np.bool_)
    y = np.zeros((n_samples, vocab_size), dtype=np.bool_)

    for i in range(n_samples):
        sent = sentences[i]

        for t in range(len(sent)):
            ch = sent[t]
            X[i, t, char_indices[ch]] = True

        target_char = next_chars[i]
        y[i, char_indices[target_char]] = True

    return X, y, char_indices, indices_char, vocab_size


def sample_next_char(probs: np.ndarray, temperature: float = 1.0) -> int:
    """Sample an index from a softmax vector (optional temperature)."""

    p = np.array(probs, dtype=np.float64)

    scaled = np.log(p + 1e-8) / temperature
    exp_p = np.exp(scaled)

    norm = exp_p / exp_p.sum()

    choice = np.random.choice(len(norm), p=norm)
    return int(choice)


def generate_continuation(
    model,
    seed_40: str,
    char_indices: dict,
    indices_char: dict,
    n_vocab: int,
    total_len: int = 400,
    temperature: float = 0.8,
) -> str:
    """Continue from a 40-char seed until total_len characters."""

    generated = seed_40
    seq_len = 40

    while True:

        if len(generated) >= total_len:
            break

        X = np.zeros((1, seq_len, n_vocab))

        window = generated[-seq_len:]

        for pos, ch in enumerate(window):
            idx = char_indices[ch]
            X[0, pos, idx] = 1

        prediction = model.predict(X, verbose=0)[0]

        next_idx = sample_next_char(prediction, temperature)
        next_character = indices_char[next_idx]

        generated = generated + next_character

    return generated


In [18]:
# -----------------------------------------------------------------------------
#  Question 2: load book, windows, GRU >=20 epochs, 400-char sample, word stats
# -----------------------------------------------------------------------------

text = load_book(Path("Book.txt"))

X, y, char_indices, indices_char, vocab_size = book_to_windows(text)

model = tf.keras.Sequential()
model.add(tf.keras.layers.GRU(128, input_shape=(40, vocab_size)))
model.add(tf.keras.layers.Dense(vocab_size, activation="softmax"))

optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.01)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy"
)

model.fit(
    X,
    y,
    epochs=20,
    batch_size=128
)

max_start = len(text) - 40
start = int(np.random.randint(0, max_start))
seed = text[start:start + 40]

generated = generate_continuation(
    model=model,
    seed_40=seed,
    char_indices=char_indices,
    indices_char=indices_char,
    n_vocab=vocab_size
)

print(generated)

words = re.findall(r"[A-Za-z]+", generated)

book_words = re.findall(r"[A-Za-z]+", text)
book_vocab = set(book_words)

count_in_book = sum(1 for w in words if w in book_vocab)

print("Total generated words:", len(words))
print("Words in book vocab:", count_in_book)


Epoch 1/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 42s 37ms/step - loss: 1.9823
Epoch 2/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 80s 75ms/step - loss: 1.6138
Epoch 3/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 49s 46ms/step - loss: 1.5248
Epoch 4/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 48s 44ms/step - loss: 1.4903
Epoch 5/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 38s 36ms/step - loss: 1.4774
Epoch 6/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 38s 35ms/step - loss: 1.4700
Epoch 7/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 38s 35ms/step - loss: 1.4688
Epoch 8/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 38s 36ms/step - loss: 1.4694
Epoch 9/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 38s 35ms/step - loss: 1.4688
Epoch 10/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 45s 42ms/step - loss: 1.4662
Epoch 11/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 47s 44ms/step - loss: 1.4716
Epoch 12/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 60s 56ms/step - loss: 1.4724
Epoch 13/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 61s 57ms/step - loss: 1.4809
Epoch 14/20
1070/1070 ━━━━━━━━━━━━━━━━━━━━ 111s 103ms/step - loss: 1.4962

---
## Question 3 — Sentiment Analysis

### Deliverables
- **(a)** Load the data, train or use **Word2Vec** on words, represent each sentence as a sequence of vectors with **padding** for variable length.
- **(b)** **60% / 40%** train / test split; **LSTM(256)**, `batch_size=64`, **sigmoid** output in [0, 1].
- **Short explanation** of how you handle different sentence lengths in a batch (e.g. padding + masking).
- **(c)** Report **test MSE** and **accuracy** where a prediction counts as correct if |ŷ − y| &lt; 0.05.

**Implement in the cells below.**

In [20]:
from sklearn.model_selection import train_test_split

def load_stanford_pairs(dict_path: Path, labels_path: Path):
    """Return (list of sentence strings, numpy label vector)."""

    sentences = []
    with open(dict_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            parts = line.split("|")
            if len(parts) > 1:
                sentences.append(parts[1])

    scores = []
    with open(labels_path, encoding="utf-8") as f:
        header = True
        for line in f:
            if header:
                header = False
                continue
            line = line.strip()
            pid, val = line.split("|")
            scores.append(float(val))

    import numpy as np
    labels = np.asarray(scores, dtype=np.float32)

    return sentences, labels


def tokenize_sentence(s: str) -> list[str]:
    """Lowercase tokenization; return a list of word tokens."""
    text = s.lower()
    words = [w for w in text.split()]
    return words


def build_padded_matrix(
    tokenized: list[list[str]], w2v: Word2Vec, max_words: int, emb_dim: int):
    """
    Tensor shape: (num_samples, max_words, emb_dim).
    Use zeros for padding / unknown words as needed.
    """

    total = len(tokenized)
    X = np.zeros((total, max_words, emb_dim), dtype="float32")

    for i in range(total):
        sent = tokenized[i]
        limit = min(len(sent), max_words)

        for j in range(limit):
            word = sent[j]

            if word in w2v.wv:
                vec = w2v.wv[word]
                X[i, j, :] = vec
            else:
                continue

    return X


def make_sentiment_model(max_words: int, emb_dim: int) -> tf.keras.Model:
    """Masking -> LSTM(256) -> Dense(1, sigmoid)."""

    inputs = tf.keras.Input(shape=(max_words, emb_dim))

    masked = tf.keras.layers.Masking(mask_value=0.0)(inputs)
    lstm_out = tf.keras.layers.LSTM(256)(masked)
    output = tf.keras.layers.Dense(1, activation="sigmoid")(lstm_out)

    model = tf.keras.Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mse"]
    )

    return model


<div style="direction:rtl">
در ابتدا جمله‌ها طول‌های متفاوتی دارند، اما شبکه عصبی برای آموزش دسته‌ای به ورودی‌هایی با طول یکسان نیاز دارد. برای حل این مشکل، طول بلندترین جمله پیدا می‌شود و همه جمله‌ها به همان طول تبدیل می‌شوند. جمله‌های کوتاه‌تر با بردارهای صفر پُر می‌شوند تا به اندازه‌ی جمله‌های بلند برسند. این فرایند همان پدینگ است.

سپس در مدل، لایه‌ای به نام ماسکینگ قرار داده می‌شود که به شبکه می‌گوید بخش‌های صفرِ اضافه‌شده واقعی نیستند و باید نادیده گرفته شوند. بنابراین LSTM فقط از کلمات واقعی استفاده می‌کند و قسمت‌های پد شده در یادگیری دخالت ندارند.

در نهایت، چون همه جمله‌ها به طول ثابت تبدیل شده‌اند، Keras می‌تواند آن‌ها را به‌صورت batchهای هم‌اندازه پردازش و مدل را آموزش دهد.

In [21]:
# -----------------------------------------------------------------------------
#  Question 3: train / evaluate — MSE and accuracy with |error| < 0.05
# -----------------------------------------------------------------------------

sentences, labels = load_stanford_pairs(
    Path("dictionary.txt"),
    Path("sentiment_labels.txt")
)

# tokenize
tokenized = []
for s in sentences:
    tokenized.append(tokenize_sentence(s))


# train word2vec
embedding_size = 100
w2v = Word2Vec(tokenized, vector_size=embedding_size, min_count=1)

emb_dim = embedding_size

# longest sentence length
lengths = [len(t) for t in tokenized]
max_words = max(lengths)

# convert to padded tensor
X_matrix = build_padded_matrix(
    tokenized,
    w2v,
    max_words,
    emb_dim
)

# split data
split = train_test_split(
    X_matrix,
    labels,
    test_size=0.4,
    random_state=42
)

X_train, X_test, y_train, y_test = split

# build model
model = make_sentiment_model(max_words, emb_dim)

# training
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# predictions
predictions = model.predict(X_test)
y_pred = predictions.reshape(-1)

# MSE calculation
errors = y_pred - y_test
test_mse = np.mean(errors ** 2)

# accuracy under threshold
threshold = 0.05
correct = np.abs(errors) < threshold
acc = np.mean(correct)

print("Test MSE:", test_mse)
print("Accuracy:", acc)

Epoch 1/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 2/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 3/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 4/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 5/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 6/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 7/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 8/10
2019/2019 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - loss: 0.0307 - mse: 0.0307 - val_loss: 0.0312 - val_mse: 0.0312
Epoch 9/10
2019/2019 ━━━━━━━━━━━━━